In [3]:
!pip install -q -U peft bitsandbytes accelerate

import glob
import json
import os
import re
import pandas as pd
import torch
from peft import PeftModel
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

test_files = glob.glob("/kaggle/input/**/test_9600.parquet", recursive=True)
adapter_files = glob.glob("/kaggle/input/**/adapter_model.safetensors", recursive=True)
assert len(test_files) > 0 and len(adapter_files) > 0, "Thiếu dataset read-splits-v1 hoặc student-grpo-adapter-v1!"

ADAPTER_DIR = os.path.dirname(adapter_files[0])
df_test = pd.read_parquet(test_files[0])

# Tập kiểm thử phân tầng cân bằng 11 miền: 550 mẫu (25 mẫu/lớp/miền)
df_test_sub = (
    df_test.groupby(["domain", "label"], group_keys=False)
    .apply(lambda g: g.sample(n=min(25, len(g)), random_state=42))
    .reset_index(drop=True)
)
print(f"Đánh giá phân tầng cân bằng 11 miền: {len(df_test_sub):,} mẫu", flush=True)

SYSTEM_PROMPT = (
    "You are an expert in textual forensics. Decide whether the given text is AI-generated or human-written. "
    "Reason step by step considering style, lexical diversity, sentence rhythm, structural symmetry, and concrete specificity. "
    "Use transition words such as 'First', 'Second', 'Moreover' and end your reasoning with your conclusion. "
    "Return STRICT JSON ONLY with exact keys: "
    '{"rationale": "3-5 concise text-specific sentences", "verdict": "AI" or "HUMAN"}.'
)

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Trải mô hình lên cả 2 GPU T4 (30GB VRAM tổng) -> Mỗi GPU chỉ dùng ~3GB!
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    dtype=torch.float16,
    device_map="auto",
)
model = PeftModel.from_pretrained(base, ADAPTER_DIR).eval()

def parse_prediction(text):
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        return None, None
    try:
        obj = json.loads(match.group(0))
        rat = str(obj.get("rationale", "")).strip()
        ver = str(obj.get("verdict", "")).strip().upper()
        if ver in {"AI", "HUMAN"} and len(rat) >= 30:
            return rat, ver
    except Exception:
        pass
    return None, None

sub_col = "sub_source" if "sub_source" in df_test_sub.columns else ("source" if "source" in df_test_sub.columns else None)
src_ds_col = "source_dataset" if "source_dataset" in df_test_sub.columns else None

results = []
BATCH_EVAL = 8
for start in tqdm(range(0, len(df_test_sub), BATCH_EVAL), desc="Fast Eval"):
    batch_df = df_test_sub.iloc[start : start + BATCH_EVAL]
    prompts = []
    for _, row in batch_df.iterrows():
        text_clip = " ".join(str(row["text"]).split()[:500])
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Analyze this text:\n\n{text_clip}"},
        ]
        prompts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=800).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=160,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    gen_tokens = outputs[:, inputs["input_ids"].shape[1] :]
    decoded = tokenizer.batch_decode(gen_tokens, skip_special_tokens=True)

    for (_, row), raw_out in zip(batch_df.iterrows(), decoded):
        rat, ver = parse_prediction(raw_out)
        results.append({
            "sample_id": int(row["sample_id"]),
            "domain": row["domain"],
            "sub_source": row[sub_col] if sub_col else "",
            "source_dataset": row[src_ds_col] if src_ds_col else "",
            "label": row["label"],
            "raw_response": raw_out.strip(),
            "rationale": rat if rat is not None else "",
            "verdict": ver if ver is not None else "INVALID",
            "pred_label": "AI" if ver == "AI" else ("Human" if ver == "HUMAN" else "INVALID"),
        })

eval_out_path = "/kaggle/working/student_grpo_test_predictions.parquet"
df_eval = pd.DataFrame(results)
df_eval.to_parquet(eval_out_path, index=False)

valid_df = df_eval[df_eval["pred_label"] != "INVALID"]
print(f"\nHOÀN TẤT! Accuracy All: {(df_eval['pred_label'] == df_eval['label']).mean():.4f} | Valid: {(valid_df['pred_label'] == valid_df['label']).mean():.4f} | FER: {1 - len(valid_df)/len(df_eval):.4f}", flush=True)

Đánh giá phân tầng cân bằng 11 miền: 550 mẫu


/tmp/ipykernel_50/1962870496.py:23: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=min(25, len(g)), random_state=42))


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Fast Eval:   0%|          | 0/69 [00:00<?, ?it/s]


HOÀN TẤT! Accuracy All: 0.3182 | Valid: 0.5452 | FER: 0.4164
